In [2]:
import requests
import pandas as pd
from IPython.display import display

In [3]:
url = "https://api.fda.gov/drug/event.json"

params = {
    "limit": 900
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("Reports retrieved:", len(data["results"]))

Reports retrieved: 900


In [4]:
reaction_terms = []

for report in data["results"]:
    patient = report.get("patient") or {}

    for reaction in patient.get("reaction") or []:
        term = reaction.get("reactionmeddrapt")

        if term:
            reaction_terms.append(term)

unique_terms = sorted(set(reaction_terms))

print("Reports loaded:", len(data["results"]))
print("Total reaction entries:", len(reaction_terms))
print("Unique reaction terms:", len(unique_terms))

display(pd.DataFrame({
    "reaction_term": unique_terms[:30]
}))

Reports loaded: 900
Total reaction entries: 2460
Unique reaction terms: 855


,reaction_term
0,Abasia
1,Abdominal discomfort
2,Abdominal distension
3,Abdominal pain
4,Abdominal pain lower
5,Abdominal pain upper
6,Abdominal tenderness
7,Abnormal behaviour
8,Abnormal dreams
9,Abnormal sensation in eye


In [5]:
reports = data["results"][:900]

rows = []

for report in reports:
    patient = report.get("patient") or {}

    reactions = patient.get("reaction") or []
    drugs = patient.get("drug") or []
    primary_source = report.get("primarysource") or {}

    rows.append({
      "report_id": report.get("safetyreportid"),
      "receivedate": report.get("receivedate"),
      "serious": report.get("serious"),
      "death_reported": report.get("seriousnessdeath"),
      "patient_age": patient.get("patientonsetage"),
      "patient_sex": {
          "1": "M",
          "2": "F",
          "0": "N/A"
      }.get(str(patient.get("patientsex")), "Unknown"),
      "reporter_country": primary_source.get(
          "reportercountry"
      ),
      "reaction_terms": "; ".join(
          r.get("reactionmeddrapt", "")
          for r in reactions
          if r.get("reactionmeddrapt")
      ),
      "drug_names": "; ".join(
          d.get("medicinalproduct", "")
          for d in drugs
          if d.get("medicinalproduct")
      ),
      "drug_roles": "; ".join(
          d.get("drugcharacterization", "")
          for d in drugs
          if d.get("drugcharacterization")
      ),
      "drug_role_description": "; ".join(
          {
              "1": "Primary Suspect Drug",
              "2": "Concomitant Drug",
              "3": "Interacting Drug"
          }.get(str(d.get("drugcharacterization")), "Unknown")
          for d in drugs
          if d.get("drugcharacterization")
      ),
      "drug_indications": "; ".join(
          d.get("drugindication", "")
          for d in drugs
          if d.get("drugindication")
      )
  })

df = pd.DataFrame(rows)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 35)

display(df.T)

0    \
report_id                                       5801206-7   
receivedate                                      20080707   
serious                                                 1   
death_reported                                          1   
patient_age                                            26   
patient_sex                                             M   
reporter_country                                   CANADA   
reaction_terms         DRUG ADMINISTRATION ERROR; OVER...   
drug_names                                  DURAGESIC-100   
drug_roles                                              1   
drug_role_description                Primary Suspect Drug   
drug_indications                               DRUG ABUSE   

                                                      1    \
report_id                                        10003300   
receivedate                                      20140306   
serious                                                 1   
death_reported                                       None   
patient_age                                            77   
patient_sex                                             F   
reporter_country                                       US   
reaction_terms         Vomiting; Diarrhoea; Arthralgia...   
drug_names                                         BONIVA   
drug_roles                                              1   
drug_role_description                Primary Suspect Drug   
drug_indications                             OSTEOPOROSIS   

                                                      2    \
report_id                                        10003301   
receivedate                                      20140228   
serious                                                 1   
death_reported                                       None   
patient_age                                          None   
patient_sex                                             F   
reporter_country                                       US   
reaction_terms                Dyspepsia; Renal impairment   
drug_names                                      IBUPROFEN   
drug_roles                                              1   
drug_role_description                Primary Suspect Drug   
drug_indications       PRODUCT USED FOR UNKNOWN INDICA...   

                                        3    \
report_id                          10003302   
receivedate                        20140312   
serious                                   2   
death_reported                         None   
patient_age                            None   
patient_sex                               M   
reporter_country                         US   
reaction_terms             Drug ineffective   
drug_names                           LYRICA   
drug_roles                                1   
drug_role_description  Primary Suspect Drug   
drug_indications                              

                                                      4    \
report_id                                        10003304   
receivedate                                      20140312   
serious                                                 2   
death_reported                                       None   
patient_age                                          None   
patient_sex                                             F   
reporter_country                                       US   
reaction_terms                      Drug hypersensitivity   
drug_names             DOXYCYCLINE HYCLATE; TRAMADOL H...   
drug_roles                      1; 1; 1; 1; 1; 1; 1; 1; 1   
drug_role_description  Primary Suspect Drug; Primary S...   
drug_indications                                            

                                                      5    \
report_id                                        10003305   
receivedate                                      20140312   
serious                                                 2   
death_reported        

In [6]:
reaction_terms = []

for report in data["results"]:
    patient = report.get("patient") or {}

    for reaction in patient.get("reaction") or []:
        term = reaction.get("reactionmeddrapt")

        if term:
            reaction_terms.append(term)

unique_terms = sorted(set(reaction_terms))

print("Total reaction entries:", len(reaction_terms))
print("Unique reaction terms:", len(unique_terms))
print("\nFirst 30 unique terms:")

display(pd.DataFrame({
    "reaction_term": unique_terms[:50]
}))

Total reaction entries: 2460
Unique reaction terms: 855

First 30 unique terms:


,reaction_term
0,Abasia
1,Abdominal discomfort
2,Abdominal distension
3,Abdominal pain
4,Abdominal pain lower
5,Abdominal pain upper
6,Abdominal tenderness
7,Abnormal behaviour
8,Abnormal dreams
9,Abnormal sensation in eye


In [7]:
print("Reports loaded:", len(data["results"]))
print("Unique reaction terms:", len(unique_terms))

Reports loaded: 900
Unique reaction terms: 855


In [8]:
# 1. Check the dataset size
print("Rows:", len(df))
print("Columns:", len(df.columns))

# 2. Check missing values
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_count"))

# 3. Check duplicate report IDs
print(
    "\nDuplicate report IDs:",
    df["report_id"].duplicated().sum()
)

Rows: 900
Columns: 12

Missing values per column:


,missing_count
report_id,0
receivedate,0
serious,0
death_reported,816
patient_age,311
patient_sex,0
reporter_country,0
reaction_terms,0
drug_names,0
drug_roles,0



Duplicate report IDs: 0


In [9]:
df = df.drop(
    columns=["patient_age", "death_reported"]
)

print(df.shape)
display(df.head())

(900, 10)


,report_id,receivedate,serious,patient_sex,reporter_country,reaction_terms,drug_names,drug_roles,drug_role_description,drug_indications
0,5801206-7,20080707,1,M,CANADA,DRUG ADMINISTRATION ERROR; OVER...,DURAGESIC-100,1,Primary Suspect Drug,DRUG ABUSE
1,10003300,20140306,1,F,US,Vomiting; Diarrhoea; Arthralgia...,BONIVA,1,Primary Suspect Drug,OSTEOPOROSIS
2,10003301,20140228,1,F,US,Dyspepsia; Renal impairment,IBUPROFEN,1,Primary Suspect Drug,PRODUCT USED FOR UNKNOWN INDICA...
3,10003302,20140312,2,M,US,Drug ineffective,LYRICA,1,Primary Suspect Drug,
4,10003304,20140312,2,F,US,Drug hypersensitivity,DOXYCYCLINE HYCLATE; TRAMADOL H...,1; 1; 1; 1; 1; 1; 1; 1; 1,Primary Suspect Drug; Primary S...,


In [10]:
# 1. Check the dataset size
print("Rows:", len(df))
print("Columns:", len(df.columns))

# 2. Check missing values
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_count"))

# 3. Check duplicate report IDs
print(
    "\nDuplicate report IDs:",
    df["report_id"].duplicated().sum()
)

Rows: 900
Columns: 10

Missing values per column:


,missing_count
report_id,0
receivedate,0
serious,0
patient_sex,0
reporter_country,0
reaction_terms,0
drug_names,0
drug_roles,0
drug_role_description,0
drug_indications,0



Duplicate report IDs: 0


In [11]:
reaction_counts = df["reaction_terms"].str.split("; ").explode()

print("Total reaction entries:", len(reaction_counts))
print("Unique reaction terms:", reaction_counts.nunique())

display(
    reaction_counts.value_counts().head(30).to_frame("count")
)

Total reaction entries: 2460
Unique reaction terms: 855


,count
reaction_terms,
Dyspnoea,51
Headache,42
Drug ineffective,39
Fatigue,38
Death,34
Pain,32
Diarrhoea,31
Malaise,30
Nausea,29


In [12]:
print(
    reaction_counts[
        reaction_counts.str.contains(
            "unevaluable|drug ineffective|overdose|error|death",
            case=False,
            na=False
        )
    ].value_counts()
)

reaction_terms
Drug ineffective                              39
Death                                         34
Overdose                                      26
Unevaluable event                             23
Drug administration error                      4
Drug prescribing error                         3
Drug dispensing error                          3
Medication error                               3
Drug ineffective for unapproved indication     2
Foetal death                                   2
DRUG ADMINISTRATION ERROR                      1
OVERDOSE                                       1
Apparent death                                 1
Accidental overdose                            1
Name: count, dtype: int64


In [13]:
print("Number of reports:", len(df))
print("Number of columns:", len(df.columns))
print("Duplicate report IDs:", df["report_id"].duplicated().sum())

Number of reports: 900
Number of columns: 10
Duplicate report IDs: 0


In [14]:
reaction_df = df.assign(
    reaction_term=df["reaction_terms"].str.split("; ")
).explode("reaction_term")

reaction_df = reaction_df[
    ["report_id", "serious", "reaction_term"]
].reset_index(drop=True)

print("Reaction entries:", len(reaction_df))
print("Unique reaction terms:", reaction_df["reaction_term"].nunique())

display(reaction_df.head(10))

Reaction entries: 2460
Unique reaction terms: 855


,report_id,serious,reaction_term
0,5801206-7,1,DRUG ADMINISTRATION ERROR
1,5801206-7,1,OVERDOSE
2,10003300,1,Vomiting
3,10003300,1,Diarrhoea
4,10003300,1,Arthralgia
5,10003300,1,Headache
6,10003301,1,Dyspepsia
7,10003301,1,Renal impairment
8,10003302,2,Drug ineffective
9,10003304,2,Drug hypersensitivity


In [15]:
display(
    reaction_df["reaction_term"]
    .value_counts()
    .head(20)
    .to_frame("count")
)

,count
reaction_term,
Dyspnoea,51
Headache,42
Drug ineffective,39
Fatigue,38
Death,34
Pain,32
Diarrhoea,31
Malaise,30
Nausea,29


In [16]:

import pandas as pd

# Work on a copy so our original dataset stays unchanged
pv_review_df = df.copy()

# Convert seriousness to a numeric value for consistent checking
pv_review_df["serious_code"] = pd.to_numeric(
    pv_review_df["serious"], errors="coerce"
)

# 1. Check whether important fields are missing or blank
required_fields = [
    "report_id",
    "receivedate",
    "serious",
    "reaction_terms",
    "drug_names"
]

pv_review_df["missing_key_data"] = False

for column in required_fields:
    missing = (
        pv_review_df[column].isna()
        | pv_review_df[column].astype("string").str.strip().eq("").fillna(True)
    )
    pv_review_df["missing_key_data"] |= missing

# 2. Flag seriousness codes outside the expected values
# Missing values are handled by the missing-data check above
pv_review_df["invalid_serious_code"] = (
    pv_review_df["serious_code"].notna()
    & ~pv_review_df["serious_code"].isin([1, 2])
)

# 3. Identify selected terms that warrant human review
review_pattern = (
    r"\b(?:death|overdose|drug ineffective|"
    r"drug administration error|unevaluable event)\b"
)

pv_review_df["selected_event_term"] = (
    pv_review_df["reaction_terms"]
    .fillna("")
    .str.contains(review_pattern, case=False, regex=True)
)

# 4. Add a transparent reason for review
# def get_review_reasons(row):
#     reasons = []

#     if row["serious_code"] == 1:
#         reasons.append("Report marked serious")

#     if row["selected_event_term"]:
#         reasons.append("Selected event term")

#     if row["missing_key_data"]:
#         reasons.append("Missing key data")

#     if row["invalid_serious_code"]:
#         reasons.append("Invalid seriousness code")

#     return reasons

# pv_review_df["review_reasons"] = pv_review_df.apply(
#     get_review_reasons, axis=1
# )

# pv_review_df["review_required"] = (
#     pv_review_df["review_reasons"].str.len() > 0
# )


def get_review_reasons(row):
    reasons = []

    if row["selected_event_term"]:
        reasons.append("Selected event term")

    if row["missing_key_data"]:
        reasons.append("Missing key data")

    if row["invalid_serious_code"]:
        reasons.append("Invalid seriousness code")

    return reasons

pv_review_df["review_reasons"] = pv_review_df.apply(
    get_review_reasons, axis=1
)

pv_review_df["review_required"] = (
    pv_review_df["review_reasons"].str.len() > 0
)




print("Reports available:", len(pv_review_df))
print("Reports flagged for review:", pv_review_df["review_required"].sum())
print("Reports with missing key data:", pv_review_df["missing_key_data"].sum())
print("Reports with invalid seriousness codes:",
      pv_review_df["invalid_serious_code"].sum())

display(
    pv_review_df.loc[
        pv_review_df["review_required"],
        ["report_id", "serious", "review_reasons"]
    ].head(10)
)


Reports available: 900
Reports flagged for review: 130
Reports with missing key data: 0
Reports with invalid seriousness codes: 0


,report_id,serious,review_reasons
0,5801206-7,1,[Selected event term]
3,10003302,2,[Selected event term]
8,10003308,2,[Selected event term]
12,10003312,1,[Selected event term]
15,10003315,1,[Selected event term]
42,10003342,1,[Selected event term]
53,10003353,2,[Selected event term]
63,10003363,2,[Selected event term]
67,10003367,2,[Selected event term]
82,10003383,2,[Selected event term]


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
import os

print("Current working directory:", os.getcwd())
print("Module exists:", os.path.exists("/content/drive/MyDrive/PV_project/pv_analysis.py"))

Current working directory: /content
Module exists: True


In [22]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os
import shutil

project_folder = "/content/drive/MyDrive/PV_project"
os.makedirs(project_folder, exist_ok=True)

shutil.copy(
    "/content/pv_analysis.py",
    os.path.join(project_folder, "pv_analysis.py")
)

print("Module saved to Google Drive:", project_folder)

FileNotFoundError: [Errno 2] No such file or directory: '/content/pv_analysis.py'

In [ ]:
print(os.listdir(project_folder))

In [ ]:
import os

drive_file = "/content/drive/MyDrive/PV_project/pv_analysis.py"

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("Project folder exists:", os.path.isdir("/content/drive/MyDrive/PV_project"))
print("Python file exists in Drive:", os.path.isfile(drive_file))

if os.path.isfile(drive_file):
    print("File size:", os.path.getsize(drive_file), "bytes")

In [24]:
%%writefile /content/drive/MyDrive/PV_project/pv_analysis.py

Overwriting /content/drive/MyDrive/PV_project/pv_analysis.py


In [25]:
import os

print("File size:", os.path.getsize(
    "/content/drive/MyDrive/PV_project/pv_analysis.py"
), "bytes")

File size: 1 bytes


In [26]:
from pathlib import Path

module_code = r'''
import pandas as pd


def flag_reports_for_review(df):
    """Flag reports for review using transparent, rule-based checks."""

    pv_review_df = df.copy()

    pv_review_df["serious_code"] = pd.to_numeric(
        pv_review_df["serious"], errors="coerce"
    )

    required_fields = [
        "report_id",
        "receivedate",
        "serious",
        "reaction_terms",
        "drug_names"
    ]

    pv_review_df["missing_key_data"] = False

    for column in required_fields:
        missing = (
            pv_review_df[column].isna()
            | pv_review_df[column]
                .astype("string")
                .str.strip()
                .eq("")
                .fillna(True)
        )
        pv_review_df["missing_key_data"] |= missing

    pv_review_df["invalid_serious_code"] = (
        pv_review_df["serious_code"].notna()
        & ~pv_review_df["serious_code"].isin([1, 2])
    )

    review_pattern = (
        r"\b(?:death|overdose|drug ineffective|"
        r"drug administration error|unevaluable event)\b"
    )

    pv_review_df["selected_event_term"] = (
        pv_review_df["reaction_terms"]
        .fillna("")
        .str.contains(review_pattern, case=False, regex=True)
    )

    def get_review_reasons(row):
        reasons = []

        if row["selected_event_term"]:
            reasons.append("Selected event term")

        if row["missing_key_data"]:
            reasons.append("Missing key data")

        if row["invalid_serious_code"]:
            reasons.append("Invalid seriousness code")

        return reasons

    pv_review_df["review_reasons"] = pv_review_df.apply(
        get_review_reasons, axis=1
    )

    pv_review_df["review_required"] = (
        pv_review_df["review_reasons"].str.len() > 0
    )

    return pv_review_df
'''

module_path = Path("/content/drive/MyDrive/PV_project/pv_analysis.py")
module_path.write_text(module_code.strip() + "\n", encoding="utf-8")

print("Module saved.")
print("File size:", module_path.stat().st_size, "bytes")

Module saved.
File size: 1792 bytes


In [27]:
import sys
import importlib

module_folder = "/content/drive/MyDrive/PV_project"

if module_folder not in sys.path:
    sys.path.insert(0, module_folder)

import pv_analysis
importlib.reload(pv_analysis)

print("Imported module from:", pv_analysis.__file__)
print("Function available:", hasattr(pv_analysis, "flag_reports_for_review"))

Imported module from: /content/drive/MyDrive/PV_project/pv_analysis.py
Function available: True


In [28]:
pv_review_df = pv_analysis.flag_reports_for_review(df)

print("Reports available:", len(pv_review_df))
print("Reports flagged for review:", pv_review_df["review_required"].sum())

Reports available: 900
Reports flagged for review: 130


In [31]:
for name in ["df", "reaction_df", "pv_review_df"]:
    print(name, "available:", name in globals())

df available: True
reaction_df available: True
pv_review_df available: True


In [30]:
output_path = "/content/drive/MyDrive/PV_project/pv_reports.csv"

df.to_csv(output_path, index=False)

print("Reports saved:", len(df))
print("File exists:", os.path.exists(output_path))

Reports saved: 900
File exists: True


Reaction Outcome Code Meaning	How to interpret it

1	Recovered / resolved	The reaction ended

2	Recovering / resolving	The reaction was improving

3	Not recovered / not resolved	The reaction had not resolved

4	Recovered with sequelae	The reaction ended, but lasting effects remained

5	Fatal	The reaction had a fatal outcome recorded

6	Unknown	The outcome was not known

• serious: 1 Yes, the adverse event report meets the definition of serious.

• serious: 2 No, the adverse event report is not classified as serious.